# numcompute_stream — Customer Churn Streaming Demo

**Assignment 2.2 | Programming for AI**

This notebook demonstrates an end-to-end streaming machine learning pipeline built entirely with **plain Python + NumPy + matplotlib** — no scikit-learn, no pandas.

### What this demo covers

| Step | Description |
|------|-------------|
| 1 | Load `dataset.csv` using the custom `io.py` pipeline |
| 2 | Split into chunks to simulate a streaming data source |
| 3 | Build three `Pipeline` instances (DecisionTree, RandomForest, AdaBoost) |
| 4 | Train each pipeline incrementally via `StreamTrainer.fit_chunk()` |
| 5 | Evaluate after every chunk via `StreamTrainer.score_chunk()` |
| 6 | Log and visualise metrics over time using `visualise.py` |
| 7 | Print the `StreamTrainer` summary table |
| 8 | Save training logs to CSV |
| 9 | Show per-chunk descriptive statistics via `update_stats()` |

## Setup

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Make the package importable when running from the demo/ folder
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from numcompute_stream import (
    Pipeline,
    StandardScaler,
    DecisionTreeClassifier,
    EnsembleClassifier,
    StreamTrainer,
    visualise,
)
from numcompute_stream.io import (
    read_csv,
    split_into_chunks,
    train_test_split,
)
from numcompute_stream.metrics import (
    Accuracy,
    confusion_matrix as batch_cm,
)
from numcompute_stream.stats import update_stats, reset_stats

# Config
N_CHUNKS    = 12
RANDOM_SEED = 42
DATA_PATH   = os.path.join("data", "dataset.csv")
OUT_DIR     = os.path.join("outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("All imports OK")

---
## Step 1 — Load the Dataset

We use `io.read_csv()` — a custom CSV parser built with NumPy — to load the customer churn dataset. No pandas.

In [ ]:
headers, data = read_csv(DATA_PATH)

X = data[:, :-1]
y = data[:, -1].astype(int)
feature_names = headers[:-1]

classes, counts = np.unique(y, return_counts=True)

print(f"File       : {DATA_PATH}")
print(f"Samples    : {X.shape[0]}")
print(f"Features   : {X.shape[1]}  →  {feature_names}")
print(f"Classes    : churn=0 (stay) → {counts[0]} customers")
print(f"             churn=1 (left) → {counts[1]} customers")
print(f"Churn rate : {counts[1] / len(y) * 100:.1f}%")

---
## Step 2 — Train/Test Split and Chunk Creation

`split_into_chunks()` divides the training data into `N_CHUNKS` equal-sized batches, simulating data arriving in a stream. The test set is held out completely and never seen during training.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

chunks = split_into_chunks(
    X_tr, y_tr, n_chunks=N_CHUNKS,
    shuffle=True, random_state=RANDOM_SEED
)

print(f"Train set  : {len(y_tr)} customers  →  {N_CHUNKS} chunks of ~{len(chunks[0][1])} each")
print(f"Test set   : {len(y_te)} customers (held out)")
print(f"Chunk sizes: {[len(yc) for _, yc in chunks]}")

---
## Step 3 — Build Three Streaming Pipelines

Each `Pipeline` chains a `StandardScaler` (using Welford's online algorithm for streaming updates) with a model. All three share the same interface — `partial_fit`, `predict`, `score` — so they can be trained and evaluated identically.

| Pipeline | Model | Key Setting |
|----------|-------|-------------|
| DecisionTree | `DecisionTreeClassifier` | `max_depth=6, criterion='gini'` |
| RandomForest | `EnsembleClassifier(method='random_forest')` | 8 trees, `sqrt(d)` features per split |
| AdaBoost | `EnsembleClassifier(method='adaboost')` | 8 boosting rounds |

In [ ]:
pipelines = {
    "DecisionTree": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  DecisionTreeClassifier(max_depth=6, criterion="gini", random_state=0)),
    ]),
    "RandomForest": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  EnsembleClassifier(
            method="random_forest", n_estimators=8, max_depth=5, random_state=0
        )),
    ]),
    "AdaBoost": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  EnsembleClassifier(
            method="adaboost", n_estimators=8, random_state=0
        )),
    ]),
}

print("Pipelines built:")
for name, pipe in pipelines.items():
    print(f"  {name}: {pipe}")

---
## Step 4 — Incremental Training via StreamTrainer

`StreamTrainer` wraps any `Pipeline` and handles all bookkeeping automatically:
- wall-clock time per chunk
- memory footprint of each incoming chunk
- training accuracy on the chunk
- evaluation accuracy on the held-out test set
- cumulative accuracy across all chunks so far

Logging is kept **separate from model logic** — the pipeline stays clean and independently testable.

In [ ]:
trainers = {
    name: StreamTrainer(pipeline=pipe, verbose=False)
    for name, pipe in pipelines.items()
}

header = f"{'Chunk':>5}  {'Customers':>10}  " + "  ".join(f"{n:>14}" for n in pipelines)
print(header)
print("─" * len(header))

for idx, (Xc, yc) in enumerate(chunks):
    row = f"{idx+1:>5}  {len(yc):>10}  "
    for name, trainer in trainers.items():
        trainer.fit_chunk(Xc, yc)
        acc = trainer.score_chunk(X_te, y_te)
        row += f"{acc:>14.4f}  "
    print(row)

---
## Step 5 — Final Evaluation on the Held-Out Test Set

In [ ]:
best_name = None
best_acc  = -1.0
final_accs = {}

for name, pipe in pipelines.items():
    acc    = pipe.score(X_te, y_te)
    preds  = pipe.predict(X_te)
    n_correct = int(np.sum(preds == y_te))
    final_accs[name] = acc
    print(f"{name:<14}: accuracy={acc:.4f}  ({n_correct}/{len(y_te)} correct)")
    if acc > best_acc:
        best_acc  = acc
        best_name = name

print(f"\nBest model: {best_name}  (acc={best_acc:.4f})")

---
## Step 6 — Visualisations

All plots use `visualise.py` — a custom matplotlib module. Each figure is displayed inline and also saved to `demo/outputs/`.

### 6a — Test Accuracy over Chunks (per model)

In [ ]:
for name, trainer in trainers.items():
    history = trainer.get_metric_history("eval_acc")
    fig = visualise.plot_metric_over_time(
        history,
        title     = f"{name} — Test Accuracy per Chunk",
        ylabel    = "Accuracy",
        xlabel    = "Chunk",
        smoothing = 3,
        save_path = os.path.join(OUT_DIR, f"{name.lower()}_accuracy.png"),
    )
    plt.show()
    plt.close(fig)

### 6b — Model Comparison: DecisionTree vs RandomForest

In [ ]:
names = list(trainers.keys())
fig = visualise.compare_models(
    trainers[names[0]].get_metric_history("eval_acc"),
    trainers[names[1]].get_metric_history("eval_acc"),
    labels    = [names[0], names[1]],
    title     = f"Model Comparison: {names[0]} vs {names[1]}",
    ylabel    = "Accuracy",
    save_path = os.path.join(OUT_DIR, "model_comparison.png"),
)
plt.show()
plt.close(fig)

### 6c — Predictions vs Ground Truth (Best Model)

In [ ]:
best_pipe = pipelines[best_name]
y_pred    = best_pipe.predict(X_te)

fig = visualise.plot_predictions_vs_ground_truth(
    y_te,
    y_pred,
    title     = f"{best_name} — Churn Predictions vs Actual (Test Set)",
    save_path = os.path.join(OUT_DIR, "predictions_vs_truth.png"),
)
plt.show()
plt.close(fig)

### 6d — Confusion Matrix

In [ ]:
mat, cls = batch_cm(y_te, y_pred)

fig = visualise.plot_confusion_matrix(
    mat,
    ["Stay (0)", "Churn (1)"],
    title     = f"{best_name} — Confusion Matrix",
    normalise = False,
    save_path = os.path.join(OUT_DIR, "confusion_matrix.png"),
)
plt.show()
plt.close(fig)

### 6e — Learning Curve (Samples Seen vs Accuracy)

In [ ]:
best_trainer  = trainers[best_name]
eval_history  = best_trainer.get_metric_history("eval_acc")
train_history = best_trainer.get_metric_history("train_acc")
n_samples_log = [
    entry["n_total"]
    for entry in best_trainer.log_
    if entry.get("eval_acc") is not None
]

fig = visualise.plot_learning_curve(
    n_samples_list = n_samples_log,
    train_scores   = train_history[:len(n_samples_log)],
    val_scores     = eval_history,
    metric_name    = "Accuracy",
    title          = f"{best_name} — Learning Curve (Customers Seen vs Accuracy)",
    save_path      = os.path.join(OUT_DIR, "learning_curve.png"),
)
plt.show()
plt.close(fig)

print(f"All plots saved to {OUT_DIR}/")

---
## Step 7 — StreamTrainer Summary Table

`summary()` prints a formatted per-chunk log showing chunk size, cumulative samples, train/eval accuracy, memory footprint, and wall-clock time. The growing time per chunk reflects the O(N) rebuild cost as the data buffer accumulates.

In [ ]:
print(f"StreamTrainer summary — best model: {best_name}\n")
trainers[best_name].summary()

---
## Step 8 — Save Training Logs to CSV

`log_to_csv()` writes the full per-chunk log for each model to `demo/outputs/<model>_log.csv`.

In [ ]:
for name, trainer in trainers.items():
    path = os.path.join(OUT_DIR, f"{name.lower()}_log.csv")
    trainer.log_to_csv(path)
    print(f"Saved: {path}")

---
## Step 9 — Streaming Descriptive Statistics via `update_stats()`

`update_stats()` in `stats.py` maintains running mean, std, min, and max incrementally using Welford's algorithm. Here we track `monthly_charges` (column index 2) as it arrives chunk by chunk — no access to future data.

In [ ]:
col_idx  = 2
col_name = feature_names[col_idx]

reset_stats()
print(f"Tracking column: '{col_name}'\n")
print(f"{'Chunk':>6}  {'Mean':>10}  {'Std':>10}  {'Min':>10}  {'Max':>10}  {'N':>6}")
print("─" * 58)

for idx, (Xc, _) in enumerate(chunks):
    col_data = Xc[:, col_idx].reshape(-1, 1)
    stats    = update_stats(col_data)

    mean_val = stats["mean"][0] if hasattr(stats["mean"], "__len__") else stats["mean"]
    std_val  = stats["std"][0]  if hasattr(stats["std"],  "__len__") else stats["std"]
    min_val  = stats["min"][0]  if hasattr(stats["min"],  "__len__") else stats["min"]
    max_val  = stats["max"][0]  if hasattr(stats["max"],  "__len__") else stats["max"]

    print(
        f"{idx+1:>6}  {mean_val:>10.2f}  {std_val:>10.2f}  "
        f"{min_val:>10.2f}  {max_val:>10.2f}  {stats['n']:>6}"
    )

---
## Summary

This notebook demonstrated a complete streaming ML workflow using **numcompute_stream** — built with NumPy only.

In [ ]:
print("══════════════════════════════════════════════════════════════")
print("  Demo Complete")
print("══════════════════════════════════════════════════════════════")
print(f"  Dataset      : Customer Churn ({len(y)} customers)")
print(f"  Features     : {feature_names}")
print(f"  Chunks       : {N_CHUNKS}")
print(f"  Train        : {len(y_tr)} customers")
print(f"  Test         : {len(y_te)} customers")
print(f"  Best model   : {best_name}  (acc={final_accs[best_name]:.4f})")
print(f"  Plots saved  : {OUT_DIR}/")